# TensorFlow: Create a custom model using subclassing approach

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import tensorflow as tf
import tensorflow_datasets as tfds

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

TF Version:  2.20.0
TF Eager mode:  True
TF GPU is available


In [2]:
class Block(tf.keras.Model):
    def __init__(self, filters, kernel_size, repetitions, pool_size=2, strides=2):
        super(Block, self).__init__()
        self.filters = filters
        self.kernel_size = kernel_size
        self.repetitions = repetitions

        # Define a conv2D_0, conv2D_1, etc based on the number of repetitions
        for i in range(repetitions):
            # Define a Conv2D layer, specifying filters, kernel_size, activation and padding.
            vars(self)[f"conv2D_{i}"] = tf.keras.layers.Conv2D(
                filters,
                kernel_size,
                activation="relu",
                padding="same")

        # Define the max pool layer that will be added after the Conv2D blocks
        self.max_pool = tf.keras.layers.MaxPool2D(
            pool_size=pool_size,
            strides=strides)

    def call(self, inputs):
        # access the class's conv2D_0 layer
        conv2D_0 = vars(self)["conv2D_0"]

        # Connect the conv2D_0 layer to inputs
        x = conv2D_0(inputs)

        # for the remaining conv2D_i layers from 1 to `repetitions` they will be connected to the previous layer
        for i in range(1, self.repetitions):
            # access conv2D_i by formatting the integer `i`. (hint: check how these were saved using `vars()` earlier)
            conv2D_i = vars(self)[f"conv2D_{i}"]
            # Use the conv2D_i and connect it to the previous layer
            x = conv2D_i(x)

        # Finally, add the max_pool layer
        max_pool = self.max_pool(x)

        return max_pool

In [3]:
class VggNetwork(tf.keras.Model):

    def __init__(self, num_classes):
        super().__init__()

        # Creating blocks of VGG with the following
        # (filters, kernel_size, repetitions) configurations
        self.block_a = Block(64, 3, 2)
        self.block_b = Block(128, 3, 2)
        self.block_c = Block(256, 3, 3)
        self.block_d = Block(512, 3, 3)
        self.block_e = Block(512, 3, 3)

        # Classification head
        # Define a Flatten layer
        self.flatten = tf.keras.layers.Flatten()
        # Create a Dense layer with 256 units and ReLU as the activation function
        self.fc = tf.keras.layers.Dense(256, activation="relu")
        # Finally add the softmax classifier using a Dense layer
        self.classifier = tf.keras.layers.Dense(num_classes, activation="softmax")

    def call(self, inputs):
        # Chain all the layers one after the other
        x = self.block_a(inputs)
        x = self.block_b(x)
        x = self.block_c(x)
        x = self.block_d(x)
        x = self.block_e(x)
        x = self.flatten(x)
        x = self.fc(x)
        x = self.classifier(x)
        return x

In [4]:
dataset = tfds.load('cats_vs_dogs', split=tfds.Split.TRAIN, data_dir='data/')

/home/denys/projects/cv-playground/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]
Generating train examples...: 0 examples [00:00, ? examples/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1771613322.471864   45878 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6979 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6

Generating train examples...: 701 examples [00:01, 700.62 examples/s]
Generating train examples...: 1778 examples [00:02, 921.57 examples/s]Corrupt JPEG data: 239 extraneous bytes before marker 0xd9
Corrupt JPEG data: 214 extraneous bytes before marker 0xd9

Generating train examples...: 2844 examples [00:03, 987.23 examples/s]Corrupt JPEG data: 128 extraneous bytes before marker 0xd9

Generating train examples...: 3931 examples [00:04, 1026.27 examples/s]
Generating train examples...: 4982 examples [00:0

Dataset cats_vs_dogs downloaded and prepared to data/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.


In [5]:
def preprocess(features):
    # Resize and normalize
    image = tf.image.resize(features['image'], (224, 224))
    return tf.cast(image, tf.float32) / 255., features['label']

dataset = dataset.map(preprocess).batch(32)

In [6]:
vgg = VggNetwork(num_classes=2)
vgg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
vgg.fit(dataset, epochs=10)

Epoch 1/10
  2/727 ━━━━━━━━━━━━━━━━━━━━ 51s 70ms/step - accuracy: 0.5781 - loss: 0.6901  

I0000 00:00:1771613363.505993   46182 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


727/727 ━━━━━━━━━━━━━━━━━━━━ 40s 49ms/step - accuracy: 0.5584 - loss: 0.6848
Epoch 2/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 32s 44ms/step - accuracy: 0.6331 - loss: 0.6499
Epoch 3/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 33s 45ms/step - accuracy: 0.6666 - loss: 0.6178
Epoch 4/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 32s 45ms/step - accuracy: 0.6808 - loss: 0.5988
Epoch 5/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - accuracy: 0.6892 - loss: 0.5870
Epoch 6/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 32s 44ms/step - accuracy: 0.6946 - loss: 0.5785
Epoch 7/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 33s 45ms/step - accuracy: 0.7006 - loss: 0.5713
Epoch 8/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 34s 47ms/step - accuracy: 0.7066 - loss: 0.5648
Epoch 9/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 34s 46ms/step - accuracy: 0.7125 - loss: 0.5587
Epoch 10/10
727/727 ━━━━━━━━━━━━━━━━━━━━ 33s 45ms/step - accuracy: 0.7177 - loss: 0.5528
